## calc def

This notebook introduces `calc def`; after running it you can define a named calculation with typed inputs and a return expression, and evaluate it against specific values.

The `timely` requirement usage from the previous notebook applies `TimelyToast` to the nominal and slow candidates. To reason about *why* the nominal candidate is appropriate, we need a quantity: the energy delivered during a toast cycle. `calc def` in SysML v2 declares a reusable calculation with typed inputs and a return expression. This notebook adds `DeliveredEnergy` to the model.

In [ ]:
import opensysml
from toaster.report import format_diagnostics

source = """package ToasterDemo {
    private import ScalarValues::*;

    abstract part def ToastingSystem {
        doc /* Transform bread into toast acceptable to its user. */
    }
    part def Heater { attribute power : Real default = 800.0; }
    part def HeatingSystem :> ToastingSystem;
    part def ControlSystem :> ToastingSystem;
    part def Toaster {
        attribute cycleTime : Real default = 120.0;
        part heating : HeatingSystem;
        part control : ControlSystem;
    }
    part nominal : Toaster;
    part slow : Toaster { attribute :>> cycleTime = 200.0; }
    requirement def TimelyToast {
        subject toaster : Toaster;
        require constraint { toaster.cycleTime <= 180.0 }
    }
    requirement timely : TimelyToast;
    part evidence {
        assert satisfy timely by nominal;
        assert satisfy timely by slow;
    }
    calc def DeliveredEnergy {
        in power : Real;
        in duration : Real;
        in efficiency : Real;
        return : Real = power * duration * efficiency;
    }
}"""

conn = opensysml.connect(version="v0.9.0")
model = conn.load_from_content(source, strict=False)
assert model.ok, f"Model failed: {format_diagnostics(model.diagnostics)}"

In [ ]:
# Negative control: a calc def that references an undefined symbol in its
# return expression raises "unresolved reference" at that site.
bad_source = """
package Bad {
    private import ScalarValues::*;
    calc def BadCalc {
        in power : Real;
        return : Real = power * undefinedEfficiency;
    }
}
"""
bad = conn.load_from_content(bad_source, strict=False)
assert not bad.ok
print("Expected error:", bad.diagnostics[0].message)

In [ ]:
result = model.eval("ToasterDemo::DeliveredEnergy(800.0, 120.0, 0.7)")
reference = 800.0 * 120.0 * 0.7  # 67200.0 J
assert abs(float(result) - reference) < 1.0, f"Unexpected: {result}"
print(f"DeliveredEnergy(800 W, 120 s, η=0.7) = {float(result):.1f} J")
conn.close()

`calc def DeliveredEnergy { in power : Real; in duration : Real; in efficiency : Real; return : Real = power * duration * efficiency; }` is the A-F expression; OpenSysML evaluates it for the given arguments (O-S); `model.eval()` returns 67200.0 J, confirming the reference value (E).

Try the chapter exercise in `exercises/ch03/exercise.ipynb`: add a `HeatLoss` calc def and verify it returns a lower effective energy for the same inputs.